# CO₂ Emission Prediction System using Regression
### Supervised Machine Learning Regression Project

**Objective:** 
এই প্রজেক্টের প্রধান উদ্দেশ্য হলো গাড়ির বিভিন্ন বৈশিষ্ট্য (যেমন: Engine Size, Cylinders, Fuel Consumption, Transmission, Fuel Type) ব্যবহার করে তার **CO₂ Emission (g/km)** কত হবে তা নির্ভুলভাবে predict বা পূর্বাভাস দেওয়া।

**Features Description:**
- **ENGINESIZE:** Engine volume in liters (L)
- **CYLINDERS:** Number of cylinders in the engine
- **FUELTYPE:** Fuel type (Z = Premium Gasoline, D = Diesel, X = Regular Gasoline, E = Ethanol)
- **VEHICLECLASS:** Class of the vehicle (Compact, SUV, Mid-size, etc.)
- **TRANSMISSION:** Transmission type (Automatic, Manual, CVT, etc.)
- **FUELCONSUMPTION_CITY:** Fuel consumption in city driving (L/100 km)
- **FUELCONSUMPTION_HWY:** Fuel consumption in highway driving (L/100 km)
- **CO2EMISSIONS:** Carbon dioxide emissions in grams per kilometer (g/km) - **Target Variable**

## Step 1: Library Imports & Environment Setup
প্রয়োজনীয় সকল লাইব্রেরি (Pandas, Numpy, Matplotlib, Seaborn, Scikit-Learn, XGBoost) ইম্পোর্ট করা হলো।

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

print("Libraries imported successfully!")

## Step 2: Data Loading & Exploration (EDA)
ডেটা ফাইল `FuelConsumptionCo2.csv` লোড করা এবং ডেটার বেসিক ইনফরমেশন ও প্রথম কয়েকটি সারি দেখা।

In [ ]:
# Load dataset
df = pd.read_csv('FuelConsumptionCo2.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
# Descriptive statistics
df.describe()

In [ ]:
# Check for missing values and data types
df.info()

### Visualizing Relationships
ফিচারগুলোর সাথে টার্গেট ভ্যারিয়েবল (CO2EMISSIONS) এর সম্পর্ক ভিজ্যুয়ালাইজ করার জন্য কিছু চার্ট তৈরি করা হলো।

In [ ]:
# Set style
sns.set_theme(style='whitegrid')

# 1. Distribution of CO2 Emissions
plt.figure(figsize=(8, 5))
sns.histplot(df['CO2EMISSIONS'], kde=True, color='#667eea')
plt.title('Distribution of CO2 Emissions')
plt.xlabel('CO2 Emissions (g/km)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# 2. Scatter Plot: Engine Size vs CO2 Emissions
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='ENGINESIZE', y='CO2EMISSIONS', hue='FUELTYPE', palette='viridis')
plt.title('Engine Size vs CO2 Emissions')
plt.xlabel('Engine Size (L)')
plt.ylabel('CO2 Emissions (g/km)')
plt.show()

In [ ]:
# 3. Correlation Matrix for Numerical Columns
plt.figure(figsize=(10, 8))
numerical_cols = ['ENGINESIZE', 'CYLINDERS', 'FUELCONSUMPTION_CITY', 'FUELCONSUMPTION_HWY', 'FUELCONSUMPTION_COMB', 'CO2EMISSIONS']
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

## Step 3: Data Preprocessing & Pipeline Definition
ফিচার এবং টার্গেট ভ্যারিয়েবল আলাদা করা এবং ক্যাটাগরিকাল ও নিউমেরিকাল ফিচারের জন্য Scikit-Learn `Pipeline` ও `ColumnTransformer` তৈরি করা হলো।

In [ ]:
target = 'CO2EMISSIONS'
categorical_features = ['FUELTYPE', 'VEHICLECLASS', 'TRANSMISSION']
numerical_features = ['ENGINESIZE', 'CYLINDERS', 'FUELCONSUMPTION_CITY', 'FUELCONSUMPTION_HWY']

X = df[numerical_features + categorical_features]
y = df[target]

# Preprocessor definition
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

# Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## Step 4: Model Training & Comparison
Linear Regression, Ridge, Decision Tree, Random Forest, এবং XGBoost মডেলগুলো ট্রেন ও টেস্ট সেটে ইভ্যালুয়েট করা হলো।

In [ ]:
def mape_score(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42)
}

results = {}

for name, model in models.items():
    # Create full pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Train model
    pipeline.fit(X_train, y_train)
    
    # Make predictions
    y_pred = pipeline.predict(X_test)
    
    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = mape_score(y_test, y_pred)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE (%)': mape, 'pipeline': pipeline}

# Display results as a DataFrame
results_df = pd.DataFrame(results).T.drop(columns=['pipeline'])
results_df

## Step 5: Visualizing Best Model's Performance
সবচেয়ে ভালো পারফরম্যান্স পাওয়া গেছে **Random Forest Regressor** দিয়ে। এই মডেলের কিছু রেসিডিউয়াল চার্ট ও অ্যাকচুয়াল বনাম প্রেডিক্টেড চার্ট নিচে আঁকা হলো।

In [ ]:
best_model_name = 'Random Forest'
best_pipeline = results[best_model_name]['pipeline']
y_pred_test = best_pipeline.predict(X_test)

# 1. Actual vs Predicted
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred_test, alpha=0.6, color='#667eea')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title(f'Actual vs Predicted CO2 Emissions ({best_model_name})', fontsize=14)
plt.xlabel('Actual CO2 Emissions (g/km)')
plt.ylabel('Predicted CO2 Emissions (g/km)')
plt.show()

In [ ]:
# 2. Residuals Plot
plt.figure(figsize=(8, 6))
residuals = y_test - y_pred_test
sns.scatterplot(x=y_pred_test, y=residuals, alpha=0.6, color='#764ba2')
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.title('Residuals vs Predicted Values', fontsize=14)
plt.xlabel('Predicted CO2 Emissions (g/km)')
plt.ylabel('Residuals (g/km)')
plt.show()

In [ ]:
# 3. Feature Importance
model_obj = best_pipeline.named_steps['model']
ohe = best_pipeline.named_steps['preprocessor'].named_transformers_['cat']
ohe_features = list(ohe.get_feature_names_out(categorical_features))
feature_names = numerical_features + ohe_features

if hasattr(model_obj, 'feature_importances_'):
    importances = model_obj.feature_importances_
    indices = np.argsort(importances)[::-1][:15]
    top_importances = importances[indices]
    top_features = [feature_names[i] for i in indices]
    
    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_importances, y=top_features, hue=top_features, legend=False, palette='viridis')
    plt.title('Top 15 Feature Importances')
    plt.xlabel('Relative Importance')
    plt.ylabel('Features')
    plt.show()

## Step 6: Model Export
ওয়েব অ্যাপ্লিকেশনে রিয়েল-টাইম প্রেডিকশনের জন্য সেরা পাইপলাইন মডেলটি সেভ করে রাখা হলো।

In [ ]:
# Save model to pickle
with open('model.pkl', 'wb') as f:
    pickle.dump(best_pipeline, f)
print("Model pipeline successfully saved to model.pkl!")